# SarcasTone - Notebook 03: Phase 1 accuracy push (GPU)

The heavy T4 experiments. On this project's CPU they cost ~25 min (DeBERTa) to ~2 h
(5-fold ensemble); on a **Colab T4** each finishes in roughly **10-15 min**.

**Protocol is unchanged and non-negotiable:** the locked test (104) is scored once per
experiment; models are selected on validation only; no test tuning.

| Exp | What | Pool | Why |
|---|---|---|---|
| E1 | 5-fold RoBERTa ensemble | locked train (482) | legitimate variance reduction on the model that won |
| E2 | 5-fold RoBERTa ensemble | expanded train (996) | does ensembling + more data help together? |
| E3 | DeBERTa-v3-base (single) | locked train (482) | stronger backbone capability probe |

Reference: champion single-split **0.6866**; 5-fold mean **0.626 ± 0.036**.

In [ ]:
import os, sys
PROJECT = '/content/SarcasTone'
if os.path.isdir(PROJECT):
    os.chdir(PROJECT)
else:
    !git clone https://github.com/PShashankreddy/SarcasTone.git $PROJECT
    os.chdir(PROJECT)
sys.path.insert(0, os.path.abspath('src'))

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('cwd =', os.getcwd())
print('device =', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '(slow - prefer a GPU runtime)')

RUN_E1 = True   # ensemble on locked train
RUN_E2 = True   # ensemble on expanded train
RUN_E3 = True   # DeBERTa probe

## Shared trainer

Reuses the repo's dataset/collate/predict so the numbers are directly comparable to the
production recipe. Selects the best-val epoch, then scores the locked test exactly once.

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          get_linear_schedule_with_warmup)
from sarcastone.models.bert_finetune import TextDataset, collate, predict
from sarcastone.training.seed import set_seed
from sarcastone.evaluation.metrics import compute_metrics
from sarcastone.evaluation.significance import mcnemar_exact, paired_bootstrap_f1
from sarcastone.utils import REPORTS_DIR, save_json

def train_one(train_df, val_df, test_df, model_name='roberta-base',
              epochs=5, lr=3e-5, batch_size=16, max_len=128, seed=42, device=DEVICE):
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    mk = lambda df, sh: DataLoader(
        TextDataset(df.text.tolist(), df.label.values, tok),
        batch_size=(batch_size if sh else 64), shuffle=sh,
        collate_fn=lambda b: collate(b, tok, max_len))
    tr, va, te = mk(train_df, True), mk(val_df, False), mk(test_df, False)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total = len(tr) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    best_f1, best_state = -1.0, None
    for ep in range(1, epochs + 1):
        model.train()
        for ids, attn, y in tr:
            out = model(input_ids=ids.to(device), attention_mask=attn.to(device), labels=y.to(device))
            opt.zero_grad(); out.loss.backward(); opt.step(); sched.step()
        yv, _, pv = predict(model, va, device)
        f1 = compute_metrics(yv, pv)['f1_macro']
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    yt, probs, pred = predict(model, te, device)
    return probs, compute_metrics(yt, pred), best_f1

def five_fold_ensemble(train_csv, test_csv, model_name, k=5, epochs=5, lr=3e-5):
    tr = pd.read_csv(train_csv)
    te = pd.read_csv(test_csv)
    y = te.label.values
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    all_probs, fold_f1 = [], []
    for fold, (ti, vi) in enumerate(skf.split(tr, tr.label), 1):
        p, m, vf1 = train_one(tr.iloc[ti], tr.iloc[vi], te, model_name,
                              epochs=epochs, lr=lr, seed=42 + fold)
        all_probs.append(p); fold_f1.append(m['f1_macro'])
        print(f'  fold {fold}: val_F1={vf1:.4f} test_F1={m["f1_macro"]:.4f}')
    mean_prob = np.mean(all_probs, axis=0)
    ens = compute_metrics(y, (mean_prob >= 0.5).astype(int))
    return {'fold_f1': fold_f1, 'mean_fold_f1': float(np.mean(fold_f1)),
            'std_fold_f1': float(np.std(fold_f1)), 'ensemble': ens,
            'mean_prob': mean_prob, 'y': y}

print('trainer ready.')

## E1 - 5-fold RoBERTa ensemble on the locked train (482)

Initialise from the headlines-boosted checkpoint (`checkpoints/text_roberta_nh`) so this is
the *same recipe* as the champion, only averaged across 5 disjoint train folds. Folds are
built from the train split only - val and test are never mixed in.

In [ ]:
if RUN_E1:
    print('E1: 5-fold ensemble, locked train (482), init=checkpoints/text_roberta_nh')
    e1 = five_fold_ensemble('data/processed/splits/train.csv',
                            'data/processed/splits/test.csv',
                            model_name='checkpoints/text_roberta_nh')
    print(f"\nE1 mean fold F1 = {e1['mean_fold_f1']:.4f} +/- {e1['std_fold_f1']:.4f}")
    print(f"E1 ensemble  F1 = {e1['ensemble']['f1_macro']:.4f}  "
          f"acc = {e1['ensemble']['accuracy']:.4f}")
    np.save(Path(REPORTS_DIR) / 'e1_ensemble_probs.npy', e1['mean_prob'])

## E2 - 5-fold RoBERTa ensemble on the expanded train (996)

Same protocol; train pool = 482 locked + 514 expansion clips (val/test still the locked 104).
Directly tests whether ensembling rescues the expansion that hurt T4a.

In [ ]:
exp_train = 'data/processed/splits_expanded/train.csv'
if RUN_E2 and os.path.exists(exp_train):
    print('E2: 5-fold ensemble, expanded train (996)')
    e2 = five_fold_ensemble(exp_train, 'data/processed/splits/test.csv',
                            model_name='checkpoints/text_roberta_nh')
    print(f"\nE2 mean fold F1 = {e2['mean_fold_f1']:.4f} +/- {e2['std_fold_f1']:.4f}")
    print(f"E2 ensemble  F1 = {e2['ensemble']['f1_macro']:.4f}  "
          f"acc = {e2['ensemble']['accuracy']:.4f}")
    np.save(Path(REPORTS_DIR) / 'e2_ensemble_probs.npy', e2['mean_prob'])
else:
    print('skipped: expanded train not found. Run 00_setup + T2/T3 build, or set RUN_E2=False.')

## E3 - DeBERTa-v3-base capability probe (locked train)

Different pretraining; if a stronger backbone does not move the locked test, that is
evidence the ceiling is data/annotation noise, not model capacity.

In [ ]:
if RUN_E3:
    tr = pd.read_csv('data/processed/splits/train.csv')
    va = pd.read_csv('data/processed/splits/val.csv')
    te = pd.read_csv('data/processed/splits/test.csv')
    print('E3: DeBERTa-v3-base, locked train (482)')
    p, m, vf1 = train_one(tr, va, te, model_name='microsoft/deberta-v3-base',
                          epochs=5, lr=2e-5, batch_size=16)
    print(f"E3 best val_F1={vf1:.4f} | test F1={m['f1_macro']:.4f} acc={m['accuracy']:.4f}")
    np.save(Path(REPORTS_DIR) / 'e3_deberta_probs.npy', p)

## Summary + significance vs the champion

Every contrast is computed on the **same locked test**. Report CIs, not just point estimates.

In [ ]:
from pathlib import Path
te = pd.read_csv('data/processed/splits/test.csv'); y = te.label.values
results = {'E1 ensemble/locked': globals().get('e1'),
           'E2 ensemble/expanded': globals().get('e2')}
rows = []
for tag, d in results.items():
    if d:
        rows.append((tag, round(d['mean_fold_f1'], 4), round(d['ensemble']['f1_macro'], 4)))
if globals().get('m'):
    rows.append(('E3 DeBERTa/locked', None, round(m['f1_macro'], 4)))
print(pd.DataFrame(rows, columns=['experiment', 'mean_fold_F1', 'test_F1']).to_string(index=False))

# significance of the best ensemble vs the champion checkpoint
ck = 'checkpoints/text_roberta_boosted'
if os.path.isdir(ck):
    tok = AutoTokenizer.from_pretrained(ck)
    model = AutoModelForSequenceClassification.from_pretrained(ck).eval().to(DEVICE)
    from sarcastone.models.bert_finetune import TextDataset as TD
    dl = DataLoader(TD(te.text.tolist(), y, tok), batch_size=64,
                    collate_fn=lambda b: collate(b, tok, 128))
    _, _, p_champ = predict(model, dl, DEVICE)
    if RUN_E1:
        pa = (e1['mean_prob'] >= 0.5).astype(int); pb = (p_champ >= 0.5).astype(int)
        print('champion F1', round(compute_metrics(y, pb)['f1_macro'], 4))
        print('E1 vs champion McNemar:', mcnemar_exact(y, pa, pb))
        print('E1 vs champion dF1    :', paired_bootstrap_f1(y, e1['mean_prob'], p_champ))

## Reading

- If the ensemble F1 sits inside the single-split CV band, it confirms the honest skill
  estimate and gives a defensible headline number with a CI.
- If DeBERTa does not beat RoBERTa here, the bottleneck is data size / annotation noise,
  not backbone capacity - a clean, reportable negative.
- Any result must be written to `reports/` and rolled into T5 (rigor suite) and T6 (gate verdict).